# 🔬 RedPitayaSTCL: Laser Locking Workflow — **Mon-Only Mode**

**Scanning Transfer Cavity Lock (STCL) using RedPitaya STEMlab 125-14**

This notebook is configured for **Mon-only** operation: the Cav RP is
replaced by a virtual `ext_scan` dummy (no real connection). The cavity
is scanned externally by a **function generator**. Only the Mon board is
physically present and connected.

---

### Board roles

| Label | Mode | Role | Physical? |
|-------|------|------|-----------|
| `Cav` | `ext_scan` | Virtual dummy — cavity scanned by function generator | ❌ No |
| `Mon` | `monitor` | Passive cavity signal monitor for live display | ✅ Yes |

<br>

<blockquote style="border-left:4px solid #2ecc71; padding:6px 12px;
  background:#eafaf1; color:#1a5c35; border-radius:4px;">
  <strong>Mon-only mode:</strong> The <code>Cav</code> entry is a software dummy
  (<code>mode='ext_scan'</code>). It satisfies all internal checks without opening
  any SSH connection or socket. The function generator drives the cavity directly.
  Phases 2–5 (Cav signal verification, cavity scan/lock, laser lock) are skipped.
</blockquote>

---

### Quick reference: scan timing (function generator)

The Mon board uses the trigger on **IN2** from the function generator's sync output.
Set `CAV_DEC` to match the function generator's scan period:

| `dec` | Sample rate | Buffer duration |
|-------|-------------|----------------|
| 8 | 15.6 MHz | 1.049 ms |
| 16 | 7.8 MHz | 2.097 ms |
| 32 | 3.9 MHz | 4.194 ms |
| 64 | 1.95 MHz | 8.389 ms |

Range and lockpoint values are always in **milliseconds**.


---
## Phase 0: Configuration

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong> → Edit this cell only </strong> before running the notebook. All parameters are passed through to the relevant phases automatically.
</blockquote>



### Board IPs
Comment out boards that are not physically present.

### Lock parameters
Pre-filled from `settings/Default.json`. Tune these after observing the cavity signal.

**Cavity range format:** `[[r1_start, r1_end], [r2_start, r2_end]]` in ms.
Two ranges required: one for each reference peak (used to measure FSR).

**Slave range format:** `[start, end]` in ms. One range per laser channel.

**PID limit:** Output voltage clamp. Keep at `[-0.99, 0.99]` initially
(maps to ±0.99 V on the output SMA). Widen only after verifying sign and stability.


In [19]:
# ── Board addresses ────────────────────────────────────────────────────────────
# Cav RP is NOT physically present — the cavity is driven by a function generator.
# A virtual ext_scan dummy named 'Cav' is created automatically below.
# Only set the Mon IP here.

# RP_CAV_IP   = "192.168.0.201"   # Not used — Cav is a virtual ext_scan dummy
# RP_LOCK1_IP = "192.168.0.102"   # Lock1 — comment out if absent
RP_MON_IP   = "192.168.0.99"    # Mon   — monitor RP (physically present)

SSH_USER = "root"
SSH_PASS = "root"


In [20]:
# ── Decimation ─────────────────────────────────────────────────────────────────
# Power of 2 between 1 and 65536. Higher = slower scan, more averaging.
# This is the ONLY place to control the scan period — do not set CAV_PERIOD_MS
# separately. The period is computed from CAV_DEC automatically.
#
# Common values:  dec=1  → 0.131 ms | dec=8  → 1.049 ms | dec=16 → 2.097 ms
#                 dec=32 → 4.194 ms | dec=64 → 8.389 ms | dec=256 → 33.6 ms
CAV_DEC = 32


In [21]:
# ── Scan signal defaults — Function Generator settings ───────────────────────
# These values describe what your function generator is outputting.
# They are stored in the dummy Cav settings for reference; the Cav RP is virtual.
#
# CAV_AMP    : half-swing of the triangle wave the FG outputs (V)
# CAV_OFFSET : DC offset of the FG output (V)
# CAV_DEC    : set this to match the FG scan period (see table in cell 0)

CAV_AMP    = 0.3   # V  — match your function generator amplitude
CAV_OFFSET = 0.0   # V  — match your function generator DC offset

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3   # ms, computed from CAV_DEC
print("Function generator scan:  amp={} V  offset={} V  dec={}  period={:.3f} ms".format(
      CAV_AMP, CAV_OFFSET, CAV_DEC, _CAV_PERIOD_MS))
print("Make sure your FG period matches the dec={} period above!".format(CAV_DEC))


Function generator scan:  amp=0.3 V  offset=0.0 V  dec=32  period=4.194 ms
Make sure your FG period matches the dec=32 period above!


In [22]:
# ── Cavity (Master) lock parameters ───────────────────────────────────────────
# Two ranges: one per reference peak. Values in ms.
CAV_RANGE     = [[0.15, 0.50], [1.70, 2.00]]
CAV_LOCKPOINT = 1.80   # ms — target position of the first reference peak
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [23]:
# ── Slave 1 lock parameters (Lock1 OUT1) ──────────────────────────────────────
SL1_LABEL     = "Laser_A"
SL1_RANGE     = [0.85, 1.10]   # ms
SL1_LOCKPOINT = 0.96           # ms
SL1_ENABLED   = True
SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [24]:
# ── Slave 2 lock parameters (Lock1 OUT2) ──────────────────────────────────────
SL2_LABEL     = "Laser_B"
SL2_RANGE     = [0.50, 0.85]   # ms
SL2_LOCKPOINT = 0.60           # ms
SL2_ENABLED   = False          # set True when second laser is coupled in
SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0,
                 "I_val": 0, "limit": [-0.99, 0.99]}

In [25]:
print("Configuration loaded (Mon-only mode).")
print("  Cav  : virtual ext_scan dummy (function generator drives cavity)")
# print("  Lock1:", RP_LOCK1_IP)
print("  Mon  :", RP_MON_IP)


Configuration loaded (Mon-only mode).
  Cav  : virtual ext_scan dummy (function generator drives cavity)
  Mon  : 192.168.0.99


---
## Phase 1: Upload & Connect

**What this does:**
1. Adds the repo root to `sys.path` so all imports resolve.
2. Creates a **virtual `Cav` dummy** (`mode='ext_scan'`) — no SSH, no socket.
   This satisfies all internal checks that look for the master RP.
3. Instantiates `LockClient` with the dummy Cav + real Mon board.
4. SSHes into Mon, uploads the RP-side scripts, loads settings.
5. Starts the PC-side selector event loop in a background thread.

> **Why a dummy Cav?** `Mon` settings store `Master = 'Cav'`. The
> `check_cavity_scanned()` guard resolves this key in `Lock.RPs`. Without
> the dummy, any function that touches Mon (including `start_monitor`) raises
> `KeyError: 'Cav'`. With `mode='ext_scan'`, the guard immediately returns
> `True` and no real communication is attempted.


In [26]:
import sys, pathlib, threading, time

# ── Locate repo root and add to path ──────────────────────────────────────────
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient_mon import LockClient, RP_client

---
## Trigger Toggle

Run **cell 12** once before `start_monitor()` to set the trigger visibility.
Dark theme, dual-axis layout, and lockpoint colours are built directly into `lockclient.py`.

| `SHOW_TRIGGER` | Effect |
|---|---|
| `True` | Two panels: IN1 cavity (top, 3×) + IN2 trigger (bottom, 1×) |
| `False` | Single panel: IN1 cavity only |

> To change after the monitor is open: stop monitor → edit flag → re-run cell 12 → restart.


In [27]:
# ── Trigger Visibility Toggle ────────────────────────────────────────────────
# Run this cell BEFORE start_monitor().
# To change while monitor is running: stop → edit SHOW_TRIGGER → re-run → restart.

from lockclient_mon import Monitor

SHOW_TRIGGER = True    # ← True = show IN2 trigger  |  False = cavity only

Monitor.show_trigger = SHOW_TRIGGER

status = "ON  — dual-axis (IN1 cavity top, IN2 trigger bottom)" if SHOW_TRIGGER else "OFF — single-axis (cavity only)"
print(f"Trigger: {status}")


Trigger: ON  — dual-axis (IN1 cavity top, IN2 trigger bottom)


In [28]:
# ── Build RP_client dictionary ────────────────────────────────────────────────
#
# 'Cav' is a virtual dummy (mode='ext_scan').
#   • No SSH connection is opened.
#   • upload_current(), start_host_server(), send() all no-op immediately.
#   • check_cavity_scanned() returns True for ext_scan → Mon monitor starts cleanly.
#
# The dummy address tuple is unused but required by RP_connection.__init__.

DUMMY_CAV_ADDR = ("0.0.0.0", 5000)   # placeholder — never connected

# Minimal Master settings for the dummy Cav.
# These mirror what a real Cav RP would have in its settings JSON.
# They are used by retrieve_settings() when Mon requests them.
_dummy_cav_settings = {
    "Master": {
        "dec": CAV_DEC,
        "range": CAV_RANGE,
        "lockpoint": CAV_LOCKPOINT,
        "enabled": True,
        "PID": CAV_PID,
        "peak_finder": {"name": "maximum", "window_size": 21, "order": 1},
    }
}

RPs = {
    "Cav": RP_client(DUMMY_CAV_ADDR, _dummy_cav_settings, mode="ext_scan"),
    # "Lock1": RP_client((RP_LOCK1_IP, 5000), {}, mode="lock"),
    "Mon": RP_client((RP_MON_IP,   5000), {}, mode="monitor"),
}

# ── Upload scripts and load settings ──────────────────────────────────────────
# LockClient.__init__ calls upload_current() on each board.
# For the Cav dummy, upload_current() is a no-op (ext_scan guard).
print("Uploading scripts to Mon and loading settings...")
Lock = LockClient(RPs)
print("Done.")
print()
print("Boards registered:")
for name, rp in Lock.RPs.items():
    print("  {:6s}  mode={:10s}  addr={}".format(name, rp.mode, rp.addr[0]))


Uploading scripts to Mon and loading settings...
Done.

Boards registered:
  Cav     mode=ext_scan    addr=0.0.0.0
  Mon     mode=monitor     addr=192.168.0.99


In [29]:
# ── Connect Mon board (starts RunLock.py via SSH) ─────────────────────────────
# The virtual Cav dummy is skipped automatically (ext_scan no-op).

def _wrap(fn, err):
    try:
        fn()
    except Exception as exc:
        err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start()
    t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(
            "{} timed out after {}s.\n"
            "Troubleshooting:\n"
            "  1. SSH into Mon and run: PYTHONPATH=/opt/redpitaya/lib/python/:$PYTHONPATH python3 /root/RunLock.py\n"
            "  2. Check port 5000 is free: ss -tlnp | grep 5000\n"
            "  3. Kill stale processes:    pkill -f RunLock.py\n"
            "  4. Power-cycle Mon if nothing else works.".format(name, timeout_s)
        )
    if "exc" in err:
        raise RuntimeError("{} failed: {}".format(name, err["exc"]))

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all (Mon only)")
print("Mon connected.")
print("Cav dummy: no connection needed (ext_scan).")


connecting...
Mon connected.
Cav dummy: no connection needed (ext_scan).


In [30]:
# ── Start PC-side event loop ──────────────────────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running.")

# ── Sync decimation on Mon ─────────────────────────────────────────────────────
# Lock.start() calls set_dec for each master RP. Since Cav is ext_scan,
# set_dec is a no-op for Cav but still updates the settings dict so Mon
# knows the correct dec for axis scaling.
# We call it explicitly here to push the Phase-0 CAV_DEC value.
Lock.set_dec("Cav", CAV_DEC)
print("Decimation set to", CAV_DEC, "— period ≈ {:.3f} ms".format(8e-9 * 16384 * CAV_DEC * 1e3))
print("Set your function generator to the same period!")

# ── Connection status ──────────────────────────────────────────────────────────
for name, rp in Lock.RPs.items():
    if rp.mode == "ext_scan":
        status = "virtual dummy (ext_scan)"
    else:
        status = "connected" if rp.connected else "DISCONNECTED"
    print("  {:6s}  {}  {}".format(name, rp.addr[0], status))


Event loop started.
Decimation set to 32 — period ≈ 4.194 ms
Set your function generator to the same period!
  Cav     0.0.0.0  virtual dummy (ext_scan)
  Mon     192.168.0.99  connected


---
## Phase 2: Signal Verification  *(Cav dummy — SKIPPED)*

<blockquote style="border-left:4px solid #95a5a6; padding:6px 12px;
  background:#f2f3f4; color:#555; border-radius:4px;">
  <strong>Skipped in Mon-only mode.</strong> Signal verification (acquiring from Cav IN1/IN2)
  requires a real Cav RP. The cells below are disabled.
  To verify the Mon signal directly, jump to <strong>Phase 3 — Live Monitor</strong>.
</blockquote>


### Matplotlib Plots

In [14]:
# ── SKIPPED: Cav signal acquisition (requires physical Cav RP) ───────────────
# The Cav dummy (ext_scan) returns None for all send() calls.
# To verify your function generator is working, connect an oscilloscope to
# the FG output and confirm the triangle waveform before running Phase 3.
print("Phase 2 skipped — Cav is a virtual dummy. No acquisition possible.")
print("Verify your function generator output on an oscilloscope before proceeding.")


Phase 2 skipped — Cav is a virtual dummy. No acquisition possible.
Verify your function generator output on an oscilloscope before proceeding.


In [15]:
# ── SKIPPED: raw Cav capture (requires physical Cav RP) ───────────────────────
print("Skipped — no physical Cav RP present.")


Skipped — no physical Cav RP present.


---
## Phase 3: Live Cavity Monitor (Mon board)

**Goal:** Start the live cavity monitor on the Mon board to observe the
cavity signal coming from your function-generator-driven scan.

**What you need:**
- Function generator output → piezo driver → cavity piezo.
- Function generator **sync / TTL output** → `Mon IN2` (trigger).
- Cavity transmission photodiode → `Mon IN1`.

**Sequence:**
1. Push Mon settings (ranges, lockpoints) from Phase 0.
2. Start the live monitor window (`start_monitor('Mon')`).
3. Adjust `CAV_RANGE` / `CAV_LOCKPOINT` in Phase 0 as needed, then re-run the settings cell.

> **If the monitor window shows a flat signal:** check that the
> function generator sync is connected to Mon IN2 (trigger) and that
> the photodiode signal is on Mon IN1.

> **Dec mismatch:** if the signal appears time-compressed or stretched,
> adjust `CAV_DEC` in Phase 0 until `_CAV_PERIOD_MS` matches the
> function generator period.


In [31]:
# ── Push initial settings to Cav dummy before starting monitor ───────────────
# Writes directly to the dummy Cav settings dict — no hardware communication.
# The monitor reads these on startup to draw range markers and lockpoint lines.

Lock.RPs["Cav"].settings["Master"]["range"]     = CAV_RANGE
Lock.RPs["Cav"].settings["Master"]["lockpoint"] = CAV_LOCKPOINT
Lock.RPs["Cav"].settings["Master"]["enabled"]   = True
Lock.RPs["Cav"].settings["Master"]["PID"]       = CAV_PID
Lock.RPs["Cav"].settings["Master"]["dec"]       = CAV_DEC
Lock.save_settings("Cav")

print("Initial settings written to Cav dummy:")
print(f"  range     : {CAV_RANGE} ms")
print(f"  lockpoint : {CAV_LOCKPOINT} ms")
print(f"  dec       : {CAV_DEC}   period ≈ {_CAV_PERIOD_MS:.3f} ms")


Initial settings written to Cav dummy:
  range     : [[0.15, 0.5], [1.7, 2.0]] ms
  lockpoint : 1.8 ms
  dec       : 32   period ≈ 4.194 ms


In [32]:
# ── Start live cavity monitor on Mon ──────────────────────────────────────────
# This opens a Qt5Agg window showing Mon IN1 in real time.
# The cavity is scanned by your function generator — no scan loop needed here.
#
# The ext_scan dummy satisfies check_cavity_scanned() → monitor starts cleanly.

Lock.start_monitor("Mon")
print("Cavity monitor started on Mon.")
print("You should see Mon IN1 updating in the Qt window.")
print()
print("To stop: Lock.stop_monitor('Mon')")


Starting background process
monitoring process started
Cavity monitor started on Mon.
You should see Mon IN1 updating in the Qt window.

To stop: Lock.stop_monitor('Mon')


In [18]:
# ── ① Edit values to update ──────────────────────────────────────────────────
# Change any values below, then run cell ② to push them to the monitor.
# All values in milliseconds.

CAV_DEC       = 32                           # decimation — must match FG period
CAV_RANGE     = [[0.2, 0.30], [1.50, 1.7]] # ms — two reference peak windows
CAV_LOCKPOINT = 1.55                         # ms — lockpoint position

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3
print(f"Values ready to push:")
print(f"  dec       : {CAV_DEC}   period ≈ {_CAV_PERIOD_MS:.3f} ms")
print(f"  range     : {CAV_RANGE}")
print(f"  lockpoint : {CAV_LOCKPOINT} ms")
print()
print("Run the next cell (②) to apply.")


Values ready to push:
  dec       : 32   period ≈ 4.194 ms
  range     : [[0.2, 0.3], [1.5, 1.7]]
  lockpoint : 1.55 ms

Run the next cell (②) to apply.


In [19]:
# ── ② Push settings to running monitor ───────────────────────────────────────
# Reads CAV_DEC / CAV_RANGE / CAV_LOCKPOINT set in cell ① above.
# Updates the monitor plot (range spans + lockpoint line) without stopping it.

# 1. Apply decimation on Mon (rescales time axis)
Lock.set_dec("Cav", CAV_DEC)
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

# 2. Write new values to the Cav dummy settings dict
Lock.RPs["Cav"].settings["Master"]["range"]     = CAV_RANGE
Lock.RPs["Cav"].settings["Master"]["lockpoint"] = CAV_LOCKPOINT
Lock.RPs["Cav"].settings["Master"]["dec"]       = CAV_DEC
Lock.save_settings("Cav")   # persist to Cav.json

# 3. Convert ms → buffer indices and push to the monitor via its queue
if Lock.monitors["Mon"]["running"].value:
    mon_settings = Lock.retrieve_monitor_settings("Cav")
    Lock.monitors["Mon"]["queue"].put(("settings", mon_settings))
    print("✓ Settings pushed to running monitor.")
else:
    print("Monitor not running — settings saved, will apply on next start_monitor().")

print(f"  dec       : {CAV_DEC}   period ≈ {_CAV_PERIOD_MS:.3f} ms")
print(f"  range     : {CAV_RANGE}")
print(f"  lockpoint : {CAV_LOCKPOINT} ms")
print()
print("Reminder: set your FG period to match {:.3f} ms".format(_CAV_PERIOD_MS))


✓ Settings pushed to running monitor.
  dec       : 32   period ≈ 4.194 ms
  range     : [[0.2, 0.3], [1.5, 1.7]]
  lockpoint : 1.55 ms

Reminder: set your FG period to match 4.194 ms


In [33]:
# ── ③ Stop monitor ────────────────────────────────────────────────────────────
Lock.stop_monitor("Mon")
print("Monitor stopped. Qt window will close.")
print()
print("To change SHOW_TRIGGER: edit cell 12 → re-run it → run cell 26 to restart.")


Monitor stopped. Qt window will close.

To change SHOW_TRIGGER: edit cell 12 → re-run it → run cell 26 to restart.


In [27]:
# ── Restart monitor ───────────────────────────────────────────────────────────
# Run after: stopping monitor, changing SHOW_TRIGGER, or changing settings.
# If you changed SHOW_TRIGGER, re-run cell 12 first.

Lock.start_monitor("Mon")
print("Monitor started.")


Starting background process
monitoring process started
Monitor started.


---
## Phase 4: Cavity Lock  *(requires physical Cav RP — SKIPPED)*

<blockquote style="border-left:4px solid #95a5a6; padding:6px 12px;
  background:#f2f3f4; color:#555; border-radius:4px;">
  <strong>Skipped in Mon-only mode.</strong> The cavity lock PID runs on the Cav RP.
  Since Cav is a virtual dummy here, cavity locking is not available.
  Your function generator maintains the scan — no PID stabilisation of the cavity length.
</blockquote>


In [ ]:
# ── SKIPPED: cavity lock (requires physical Cav RP) ──────────────────────────
print("Skipped — Cav is virtual. Cavity lock not available in Mon-only mode.")


In [ ]:
# ── SKIPPED: start cavity lock ────────────────────────────────────────────────
print("Skipped — Cav is virtual.")


---
## Phase 5: Laser Lock  *(requires Lock1 board — SKIPPED)*

<blockquote style="border-left:4px solid #95a5a6; padding:6px 12px;
  background:#f2f3f4; color:#555; border-radius:4px;">
  <strong>Skipped in Mon-only mode.</strong> Lock1 is not present in this configuration.
  Add Lock1 back to the RPs dict when it becomes available.
</blockquote>


In [ ]:
# ── Check Lock1 is available ───────────────────────────────────────────────────
if "Lock1" not in Lock.RPs:
    print("Lock1 not in RPs dict — skipping laser lock phase.")
    print("Add Lock1 to the BOARDS config in Phase 0 and restart from Phase 1.")
else:
    print("Lock1 found:", Lock.RPs["Lock1"].addr[0])
    print("Connected  :", Lock.RPs["Lock1"].connected)

In [ ]:
# ── Push Slave1 settings ───────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)
    print("Slave1 ({}) settings pushed:".format(SL1_LABEL))
    print("  range     :", SL1_RANGE, "ms")
    print("  lockpoint :", SL1_LOCKPOINT, "ms")
    print("  enabled   :", SL1_ENABLED)
    print("  PID       :", SL1_PID)

In [ ]:
# ── Push Slave2 settings (only if SL2_ENABLED = True) ─────────────────────────
if "Lock1" in Lock.RPs:
    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)
    print("Slave2 ({}) settings pushed  (enabled={})".format(SL2_LABEL, SL2_ENABLED))

In [ ]:
# ── Start laser lock ───────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs:
    Lock.start_lock("Lock1")
    print("Laser lock started on Lock1.")
    print("Stop with: Lock.stop_loop('Lock1')")

---
## Phase 6: Error Monitor  *(Mon board)*

Opens a live matplotlib window showing the **frequency error** of each
laser channel in MHz over time. In Mon-only mode this is useful for
watching the Mon signal quality (once a lock is active via the Cav RP).

> In Mon-only mode with a function generator, the cavity is not actively
> locked, so error values will drift freely. This monitor is most useful
> once the full system is restored with the physical Cav RP.


In [ ]:
# ── Start cavity signal monitor (Mon board) ────────────────────────────────────
if "Mon" in Lock.RPs:
    Lock.start_monitor("Mon")
    print("Cavity monitor started.")
else:
    print("Mon board not present — skipping cavity monitor.")
    print("You can still watch errors with the error monitor below.")

In [ ]:
# ── Start error monitor (Mon board) ───────────────────────────────────────────

# tmin: minimum time between error samples in seconds (default 10 ms).
# Opens a live plot window. Runs until Lock.stop_monitor("Mon") is called.

if "Mon" in Lock.RPs:
    Lock.stop_monitor("Mon")          # stop cavity monitor first if running
    time.sleep(0.5)
    Lock.start_error_monitor("Mon", tmin=20e-3)
    print("Error monitor started. Close the plot window or run the next cell to stop.")
else:
    print("Mon board not present — skipping error monitor.")

In [ ]:
# ── Save error data ────────────────────────────────────────────────────────────
# Saves the recorded errors to a JSON file. Run while monitor is still open.
# The file will contain time axis + per-channel error arrays.

# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# if "Mon" in Lock.RPs:
#     Lock.monitors["Mon"]["queue_err"].put(("save", filename))
#     print("Saving errors to {}.json".format(filename))

---
## Phase 7: Safe Shutdown

**Always shut down in this order:**
1. Stop error / cavity monitors first.
2. Stop laser lock loops (`Lock1`) before stopping the cavity scan/lock.
3. Stop cavity loop last — the cavity RP generates the trigger for all others.
4. Disconnect all RPs (closes socket servers on boards).
5. Stop the PC-side event loop.

Skipping this order can leave boards with stale processes holding ports
5000 / 5065, requiring a manual `pkill` on the board before the next session.

> `Lock.close()` performs all steps above in the correct order automatically.
> Use the individual cells below only when you need fine-grained control
> (e.g. stopping one laser lock while keeping the cavity lock running).


In [34]:
# ── Stop monitors ──────────────────────────────────────────────────────────────
if "Mon" in Lock.RPs:
    Lock.stop_monitor("Mon")
    time.sleep(0.5)
    print("Monitor stopped.")

Monitor stopped.


In [35]:
# ── Stop laser lock ────────────────────────────────────────────────────────────
if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
    Lock.stop_loop("Lock1")
    time.sleep(0.5)
    print("Laser lock stopped.")

In [36]:
# ── Stop cavity dummy (no-op — ext_scan never runs a loop) ──────────────────
# The Cav dummy has no running loop. This cell is kept for completeness.
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    print("Cav loop stopped.")
else:
    print("Cav dummy: no loop was running (expected in Mon-only mode).")


Cav dummy: no loop was running (expected in Mon-only mode).


In [37]:
# ── Full clean shutdown ────────────────────────────────────────────────────────
# Stops all loops, disconnects all boards, stops the event loop.
# Run this at the end of every session.

Lock.close()
print("All boards disconnected. Session closed.")
print()
print("Board port status after shutdown:")
print("  Ports 5000 and 5065 should now be free on all boards.")
cmd = 'ss -tlnp | grep -E "5000|5065"'
print(f"  Verify with:  ssh root@<IP> '{cmd}'")


Cav not connected.
All boards disconnected. Session closed.

Board port status after shutdown:
  Ports 5000 and 5065 should now be free on all boards.
  Verify with:  ssh root@<IP> 'ss -tlnp | grep -E "5000|5065"'
